# TinyForge Vision - Training on Colab

## Introduction

This notebook trains **TinyForge Vision**, a 222M-parameter vision-language model (**SigLIP-B/16-224** + **SmolLM2-135M**), on the preprocessed Cauldron and scores it on MMStar. The target is **nanoVLM v0.1**, whose published checkpoint (`lusxvr/nanoVLM-222M`) reports 35.3% on MMStar.

- **Checks** before the full run: the published checkpoint scored with our eval, the fast tests, a GPU attention check, overfitting one batch and a 100-step run.
- **Training**: 5 epochs (32,330 steps) at batch size 256, with validation loss and MMStar (first 500 questions) every 1,000 steps.
- **Evaluation**: full MMStar (1,500 questions, 3 seeds) for `best_mmstar` and `final`, sample generations, and the validation-loss checks behind the gap explained in `WRITEUP.md`.

The run below used one A100-SXM4 80 GB. It needs two Colab secrets: `HF_TOKEN` (read access to the preprocessed dataset) and `WANDB_API_KEY`.

## Setup

Mount Google Drive so checkpoints survive a disconnect, clone the repository and install it, then log in to Hugging Face and W&B. `REPO_URL` and `CHECKPOINT_DIR` are the only values to edit.

In [1]:
!nvidia-smi

Mon Nov 16 13:12:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   44C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Edit these two before running.
REPO_URL = "https://github.com/JuaniLlaberia/tinyforge-vision.git"
CHECKPOINT_DIR = "/content/drive/MyDrive/tinyforge-vision/checkpoints"

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!git clone {REPO_URL} /content/tinyforge-vision
%cd /content/tinyforge-vision
!pip install -q uv && uv pip install --system -q -e . pytest

Cloning into '/content/tinyforge-vision'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 81 (delta 20), reused 81 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 263.55 KiB | 16.47 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/tinyforge-vision


In [4]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: juanillaberia (juanillaberia-punto-medio) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Checks

### Fast Tests

CPU tests that compare the projector, forward pass, collator and generation against the v0.1 reference code in `tests/reference/`. All must pass before spending GPU time; rerun them after any code change.

In [7]:
!python -m pytest -m "not slow" -q

.....................                                                    [100%]
21 passed, 3 deselected in 13.00s


### Baseline Eval

The published checkpoint scored with this repository's MMStar evaluation: the same code, settings and seeds used for our model at the end.

In [ ]:
!python scripts/eval_mmstar.py --impl ours --seeds 0 1 2

config.json: 100% 903/903 [00:00<00:00, 5.27MB/s]
model.safetensors: 100% 888M/888M [00:03<00:00, 247MB/s]
tokenizer_config.json: 3.91kB [00:00, 1.98MB/s]
vocab.json: 801kB [00:00, 87.2MB/s]
merges.txt: 466kB [00:00, 132MB/s]
tokenizer.json: 2.10MB [00:00, 180MB/s]
special_tokens_map.json: 100% 489/489 [00:00<00:00, 3.39MB/s]
README.md: 4.22kB [00:00, 16.6MB/s]
mmstar.parquet: 100% 41.8M/41.8M [00:01<00:00, 41.6MB/s]
Generating val split: 100% 1500/1500 [00:00<00:00, 12378.08 examples/s]
ours | lusxvr/nanoVLM-222M | 1500 questions | seeds [0, 1, 2] | cuda
  seed 0: 0.3787 (78s)
  seed 1: 0.3860 (78s)
  seed 2: 0.3680 (78s)
MMStar: 0.3776 ± 0.0091 over 3 seeds (published nanoVLM-222M: 0.353)


The published checkpoint scores **37.8% ± 0.9** over the 3 seeds, 2.5 points above the reported 35.3%. The evaluation samples its answers, so the seeds alone span 1.8 points. Since both models are scored with exactly the same code, 37.8% is the fair baseline for ours.

### GPU Attention Check

The language model merges the causal and padding masks into a single `attn_mask`, because recent PyTorch rejects `attn_mask` together with `is_causal=True`. This runs the published checkpoint on one padded batch in fp32 and in bf16: both must run, and their losses must agree closely.

In [8]:
import torch
from tfv.data.collators import VQACollator
from tfv.data.processors import get_tokenizer
from tfv.models.vlm import VisionLanguageModel

model = VisionLanguageModel.from_pretrained("lusxvr/nanoVLM-222M").cuda().eval()
tokenizer = get_tokenizer(model.cfg.lm_tokenizer)
samples = [
    {"image": torch.rand(3, 224, 224), "text_data": "Question: What is this? Answer:", "answer": "A cat." + tokenizer.eos_token},
    {"image": torch.rand(3, 224, 224), "text_data": "Question: How many people are standing near the red door? Answer:",
     "answer": "There are three people." + tokenizer.eos_token},
]
batch = {k: v.cuda() for k, v in VQACollator(tokenizer, model.cfg.lm_max_length)(samples).items()}
args = (batch["input_ids"], batch["image"], batch["attention_mask"])

with torch.no_grad():
    _, loss_fp32 = model(*args, targets=batch["labels"])
    with torch.autocast("cuda", dtype=torch.bfloat16):
        _, loss_bf16 = model(*args, targets=batch["labels"])
print(f"fp32 loss {loss_fp32.item():.4f} | bf16 loss {loss_bf16.item():.4f}")
assert torch.isfinite(loss_bf16) and abs(loss_fp32.item() - loss_bf16.item()) < 0.1
del model
torch.cuda.empty_cache()

fp32 loss 3.9910 | bf16 loss 3.9975


### Configuration

Defaults live in `tfv/config.py`; this cell only sets the checkpoint folder, `micro_batch_size`, the data-loader workers and the eval schedule. The batch is 256 = `micro_batch_size` x gradient accumulation. A micro-batch of 256 fits on an 80 GB GPU (47 GB peak in this run); lower it (128, 64) if memory runs out.

In [12]:
import dataclasses
from tfv.config import EvalConfig, TrainConfig, VLMConfig
from tfv.train.train import train

vlm_cfg = VLMConfig()
eval_cfg = EvalConfig()
train_cfg = TrainConfig(
    checkpoint_dir=CHECKPOINT_DIR,
    micro_batch_size=256,
    num_workers=10,
    eval_interval=1000,
    mmstar_limit=500,
)

print(train_cfg)

TrainConfig(lr_mp=0.002, lr_backbones=0.0001, weight_decay=0.01, betas=(0.9, 0.999), max_grad_norm=None, warmup_frac=0.03, min_lr_frac=0.1, freeze_vision_encoder=False, freeze_language_model=False, batch_size=256, micro_batch_size=256, epochs=5, max_steps=None, compile=True, train_dataset_path='Juanillaberia/tinyforge-vision-cauldron-224', train_dataset_name=('ai2d', 'aokvqa', 'chart2text', 'chartqa', 'clevr', 'cocoqa', 'datikz', 'diagram_image_to_text', 'docvqa', 'dvqa', 'figureqa', 'finqa', 'geomverse', 'hateful_memes', 'hitab', 'iam', 'iconqa', 'infographic_vqa', 'intergps', 'localized_narratives', 'mapqa', 'multihiertt', 'ocrvqa', 'plotqa', 'raven', 'rendered_text', 'robut_sqa', 'robut_wikisql', 'robut_wtq', 'scienceqa', 'screen2words', 'st_vqa', 'tabmwp', 'tallyqa', 'tat_qa', 'textcaps', 'textvqa', 'tqa', 'vistext', 'visual7w', 'visualmrc', 'vqarad', 'vqav2', 'vsr', 'websight'), data_cutoff_idx=None, val_ratio=0.01, num_workers=10, seed=0, eval_interval=1000, mmstar_limit=500, num

### Overfit One Batch

300 steps on a single batch. The loss should fall close to zero; if it doesn't, a module or the labels are wrong. The first data load also downloads the preprocessed Cauldron (~30 GB) into the runtime's cache, and later cells reuse it.

In [10]:
overfit_cfg = dataclasses.replace(train_cfg, overfit_batch=True, max_steps=300, eval_interval=0, mmstar_limit=100,
                                  log_wandb=False, run_name="check-overfit", log_interval=25)
result = train(overfit_cfg, VLMConfig(), eval_cfg)
print(f"Final overfit loss: {result['train_loss']:.4f} (expect well below 0.5; if it stalls above ~1, check the modules and labels)")

Loading from backbone weights


Successfully loaded google/siglip-base-patch16-224 weights from safetensors. Model has 85,797,120 parameters.


Successfully loaded HuggingFaceTB/SmolLM2-135M weights from safetensors. Model has 134,515,008 parameters.
TinyForge-Vision initialized with 222,081,600 parameters
check-overfit: 1,655,332 train / 16,720 val samples | 6466 steps/epoch, 300 total | batch 256 = 1 x 256 | cuda, torch.bfloat16


epoch 1 | step     25/300 | loss 0.6054 | grad norm 9.53 | lr 1.99e-03 | tok/s 52,710 | elapsed 2.5 min
epoch 1 | step     50/300 | loss 0.0034 | grad norm 0.18 | lr 1.92e-03 | tok/s 52,483 | elapsed 2.6 min
epoch 1 | step     75/300 | loss 0.0008 | grad norm 0.02 | lr 1.79e-03 | tok/s 52,287 | elapsed 2.8 min
epoch 1 | step    100/300 | loss 0.0005 | grad norm 0.01 | lr 1.61e-03 | tok/s 52,331 | elapsed 3.0 min
epoch 1 | step    125/300 | loss 0.0004 | grad norm 0.01 | lr 1.39e-03 | tok/s 52,221 | elapsed 3.2 min
epoch 1 | step    150/300 | loss 0.0003 | grad norm 0.00 | lr 1.15e-03 | tok/s 52,496 | elapsed 3.3 min
epoch 1 | step    175/300 | loss 0.0003 | grad norm 0.00 | lr 9.12e-04 | tok/s 52,330 | elapsed 3.5 min
epoch 1 | step    200/300 | loss 0.0003 | grad norm 0.00 | lr 6.84e-04 | tok/s 52,413 | elapsed 3.7 min
epoch 1 | step    225/300 | loss 0.0003 | grad norm 0.00 | lr 4.86e-04 | tok/s 52,264 | elapsed 3.9 min
epoch 1 | step    250/300 | loss 0.0002 | grad norm 0.00 | lr 3.

The loss reaches **0.0002**. Validation loss (4.59) and MMStar (3%) are bad, as expected for a model that memorized one batch.

### 100-Step Run

A short run with the full configuration. Training and validation loss should both fall. `train/peak_mem_gb` and `train/samples_per_second` in W&B give the numbers to budget the full run.

In [11]:
check_cfg = dataclasses.replace(train_cfg, max_steps=100, eval_interval=50, mmstar_limit=200, run_name="check-100steps")
result = train(check_cfg, VLMConfig(), eval_cfg)
result

Loading from backbone weights
Successfully loaded google/siglip-base-patch16-224 weights from safetensors. Model has 85,797,120 parameters.
Successfully loaded HuggingFaceTB/SmolLM2-135M weights from safetensors. Model has 134,515,008 parameters.
TinyForge-Vision initialized with 222,081,600 parameters


check-100steps: 1,655,332 train / 16,720 val samples | 6466 steps/epoch, 100 total | batch 256 = 1 x 256 | cuda, torch.bfloat16
  eval @ step 0: val loss 3.9804 | MMStar 0.2050 (best, saved) | 21s
    'Question: Is the window on the back of the black truck tinted?\nBe succinct. Answer:' -> ' Not unlike our feminist president of a long established Jewish community, "Who does that long box shaped vehicle' (reference 'Yes.')
    'Question: Describe this image in one or two sentences. Answer:' -> ' Houdina, Issue number one.55( Laughter). Vamp und ago 5' (reference 'In this image we can see a boy wearing a crown. In the background of the image there are people. At the bottom of the image there is floor.')
    'Question: Which algorithm has the highest accuracy?\nMake the answer very short. Answer:' -> " Lüft's algorithm: mainstream\nmemory banks are 18 billion bits long. Thus," (reference 'Sera.')


  eval @ step 50: val loss 2.0394 | MMStar 0.0300 | 19s
    'Question: Is the window on the back of the black truck tinted?\nBe succinct. Answer:' -> 'Yes.Stasi was the nickname of the infamous regime.Jike Doepke,' (reference 'Yes.')
    'Question: Describe this image in one or two sentences. Answer:' -> 'The board overlooking an apple orchard is battered with sand and grass. On the background there are few' (reference 'In this image we can see a boy wearing a crown. In the background of the image there are people. At the bottom of the image there is floor.')
    'Question: Which algorithm has the highest accuracy?\nMake the answer very short. Answer:' -> 'Eagle.कहें तक डॉसिल' (reference 'Sera.')
epoch 1 | step    100/100 | loss 1.9653 | grad norm 1.49 | lr 2.00e-04 | tok/s 52,891 | elapsed 1.1 min
  eval @ step 100: val loss 1.9761 | MMStar 0.0400 | 19s
    'Question: Is the window on the back of the black truck tinted?\nBe succinct. Answer:' -> 'No.This diode is a digital CMOS PQMTL 

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▄█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,██▆▅▃▃▃▂▃▂▃▁▂▂▂▁▂▂▂▂▁▂▂▁▁▁▁▁▂▁▁▂▂▁▁▁▁▁▁▁
train/lr_group0,██████▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
train/lr_group1,▃████████▇▇▇▇▇▆▆▆▆▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
train/peak_mem_gb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/samples_per_second,▁█████▇███████████▇██████████████████▇██
train/tokens_per_second,█▅▇▆█▄▇▁▂▅▁▇▆▇█▄▃▅█▆▇▃▅▆▆▄▆▄▅█▇▃█▂▃▆▃▆▄█
val/best_mmstar,▁▁▁
+3,...


{'step': 100,
 'train_loss': 1.9653110496193158,
 'val_loss': 1.9760550740015697,
 'mmstar': 0.04,
 'run_dir': '/content/drive/MyDrive/tinyforge-vision/checkpoints/check-100steps'}

Validation loss falls from 3.98 to 1.98, and the answers start with the right kind of answer (`Yes.`, `No.`) before running on. MMStar (first 200 questions) drops from 20.5% to 4% this early; in the full run it is back above its starting score by step 1,000.

## Training

The full run: 5 epochs, 32,330 steps. A resumable `latest.pt` (model and optimizer) is written to Drive every 1,000 steps, and `best_mmstar/` and `final/` are saved models. `RUN_NAME` is fixed so a resumed run finds its folder.

In [13]:
RUN_NAME = "tfv-full-v1"

full_cfg = dataclasses.replace(train_cfg, run_name=RUN_NAME)
result = train(full_cfg, VLMConfig(), eval_cfg)
result

Loading from backbone weights
Successfully loaded google/siglip-base-patch16-224 weights from safetensors. Model has 85,797,120 parameters.
Successfully loaded HuggingFaceTB/SmolLM2-135M weights from safetensors. Model has 134,515,008 parameters.
TinyForge-Vision initialized with 222,081,600 parameters


tfv-full-v1: 1,655,332 train / 16,720 val samples | 6466 steps/epoch, 32330 total | batch 256 = 1 x 256 | cuda, torch.bfloat16
  eval @ step 0: val loss 3.9804 | MMStar 0.1720 (best, saved) | 26s
    'Question: Is the window on the back of the black truck tinted?\nBe succinct. Answer:' -> " It's not all bad\nPoor duh...\ndesire/patience/levity/" (reference 'Yes.')
    'Question: Describe this image in one or two sentences. Answer:' -> 'Where are you?TLB:Replace participants with uk store transaction rate with the following:1' (reference 'In this image we can see a boy wearing a crown. In the background of the image there are people. At the bottom of the image there is floor.')
    'Question: Which algorithm has the highest accuracy?\nMake the answer very short. Answer:' -> '   `LCP algorithm calculated, but Egypt doesn’t let You don’t bat an Eye' (reference 'Sera.')


epoch 1 | step    100/32330 | loss 2.4065 | grad norm 21.25 | lr 2.04e-04 | tok/s 49,934 | elapsed 0.8 min
epoch 1 | step    200/32330 | loss 1.9652 | grad norm 11.09 | lr 4.11e-04 | tok/s 50,639 | elapsed 1.5 min
epoch 1 | step    300/32330 | loss 2.0284 | grad norm 7.67 | lr 6.17e-04 | tok/s 50,332 | elapsed 2.2 min
epoch 1 | step    400/32330 | loss 1.8107 | grad norm 4.80 | lr 8.24e-04 | tok/s 49,662 | elapsed 2.9 min
epoch 1 | step    500/32330 | loss 1.7740 | grad norm 4.47 | lr 1.03e-03 | tok/s 52,076 | elapsed 3.6 min
epoch 1 | step    600/32330 | loss 1.8850 | grad norm 3.95 | lr 1.24e-03 | tok/s 49,070 | elapsed 4.3 min
epoch 1 | step    700/32330 | loss 1.6664 | grad norm 2.94 | lr 1.44e-03 | tok/s 50,029 | elapsed 5.1 min
epoch 1 | step    800/32330 | loss 1.8372 | grad norm 3.31 | lr 1.65e-03 | tok/s 52,643 | elapsed 5.8 min
epoch 1 | step    900/32330 | loss 1.5681 | grad norm 3.67 | lr 1.86e-03 | tok/s 51,946 | elapsed 6.5 min
epoch 1 | step   1000/32330 | loss 1.5691 | 

epoch,▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆███████
step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▆▅▅▅▄▅▄▃▄▃▃▃▃▃▃▃▃▂▂▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
train/lr_group0,▁█████▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▁▁▁▁▁▁▁▁▁
train/lr_group1,▆████▇▇▇▇▇▆▆▆▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▂▂▂▁▁▁▁▁
train/peak_mem_gb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/samples_per_second,▅▇▅▅▃▅███▆▇████▅██▇█▅██▂▄█▅▇▇█▃▇██▆▅▇▇▅▁
train/tokens_per_second,▆▆▄▇▄▅▆▄▇▃▇▆▂▄▄▄▂▅▆▇▄▆▇▆▃▆▅▃▃▇█▄▅▇▁▆▄▃▆▄
val/best_mmstar,▁▂▂▆▆▆▇▇▇▇▇▇██████████████████████
+3,...


{'step': 32330,
 'train_loss': 0.591029124155594,
 'val_loss': 0.9916660135100611,
 'mmstar': 0.198,
 'run_dir': '/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1'}

The run finished in a single session of about 4.2 hours, with a final train loss of 0.59 and a validation loss of **0.99**. MMStar on the first 500 questions peaked at 29.6% at step 12,000 (saved as `best_mmstar`) and ended at 19.8%. The curves are in `WRITEUP.md` and W&B.

### Resume

Only needed after a disconnect (not used for this run). Rerun Setup and the Configuration cell, then this cell: it continues from the last `latest.pt` with the same data order and the same W&B run.

In [ ]:
resume_cfg = dataclasses.replace(train_cfg, run_name=RUN_NAME, resume_from=f"{CHECKPOINT_DIR}/{RUN_NAME}/latest.pt")
result = train(resume_cfg, VLMConfig(), eval_cfg)
result

## Evaluation

### MMStar

Full MMStar (1,500 questions) over 3 seeds for both saved models, with the same code and seeds as the baseline.

In [14]:
!python scripts/eval_mmstar.py --impl ours --model {CHECKPOINT_DIR}/{RUN_NAME}/final --seeds 0 1 2 --output {CHECKPOINT_DIR}/{RUN_NAME}/mmstar_final.json

!python scripts/eval_mmstar.py --impl ours --model {CHECKPOINT_DIR}/{RUN_NAME}/best_mmstar --seeds 0 1 2 --output {CHECKPOINT_DIR}/{RUN_NAME}/mmstar_best.json

ours | /content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/final | 1500 questions | seeds [0, 1, 2] | cuda
  seed 0: 0.2273 (39s)
  seed 1: 0.2160 (35s)
  seed 2: 0.2100 (36s)
MMStar: 0.2178 ± 0.0088 over 3 seeds (published nanoVLM-222M: 0.353)
ours | /content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/best_mmstar | 1500 questions | seeds [0, 1, 2] | cuda
  seed 0: 0.2687 (36s)
  seed 1: 0.2887 (35s)
  seed 2: 0.2713 (35s)
MMStar: 0.2762 ± 0.0109 over 3 seeds (published nanoVLM-222M: 0.353)


`best_mmstar` scores **27.6% ± 1.1** and `final` **21.8% ± 0.9**, against 37.8% for the published checkpoint. More training kept lowering validation loss but hurt MMStar, and even the best checkpoint is 10 points behind.

### Sample Generations

Five sampled answers from `final` to "What is this?" about nanoVLM's example image (a cat standing on pebbles). The image isn't in the repository, so the cell downloads it.

In [15]:
!mkdir -p assets && test -f assets/image.png || curl -sL -o assets/image.png https://raw.githubusercontent.com/huggingface/nanoVLM/v0.1/assets/image.png

!python scripts/generate.py --checkpoint {CHECKPOINT_DIR}/{RUN_NAME}/final --prompt "What is this?"

Using device: cuda
Loading weights from: /content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/final

Input:
  What is this? 

Outputs:
  >> Generation 1: This picture might be taken inside the room. In this image, in the middle, we can see
  >> Generation 2: This picture contains a white and brown color cat. It is facing towards the left side of the picture
  >> Generation 3: This image is taken outdoors. At the bottom of the image there is a ground with pebbles on it
  >> Generation 4: This image is a depiction. In the middle of the image there is a cat, keeping it's
  >> Generation 5: In the center of the picture there is a black cat, it is standing on the pebbles. On


Every answer describes the cat and the pebbles, in the style of `localized_narratives` ("In the center of the picture there is..."), one of the largest Cauldron subsets. Colors are less reliable: the same cat is white and brown in one answer and black in another.

### Validation Loss

Loss on the held-out 1% of the Cauldron for the published checkpoint and both of ours.

In [16]:
import torch
from tfv.config import VLMConfig
from tfv.data.loaders import get_train_val_datasets, get_train_val_loaders
from tfv.models.vlm import VisionLanguageModel
from tfv.train.train import evaluate_val_loss

device = torch.device("cuda")
_, val_ds = get_train_val_datasets(train_cfg, VLMConfig())
_, val_loader = get_train_val_loaders(val_ds, val_ds, train_cfg, VLMConfig())

for path in ["lusxvr/nanoVLM-222M", f"{CHECKPOINT_DIR}/{RUN_NAME}/best_mmstar", f"{CHECKPOINT_DIR}/{RUN_NAME}/final"]:
    model = VisionLanguageModel.from_pretrained(path).to(device)
    print(f"{path}: val loss {evaluate_val_loss(model, val_loader, device, torch.bfloat16, True):.4f}")

lusxvr/nanoVLM-222M: val loss 1.8973
/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/best_mmstar: val loss 1.0538
/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/final: val loss 0.9917


Ours fits the Cauldron far better than the published checkpoint (**0.99** against **1.90**), the opposite of what a weaker training run would show. The next cells look for the cause.

### Multiple-Choice Answers

Raw outputs on the first 16 MMStar questions (seed 0), to see how each model answers.

In [17]:
from tfv.config import EvalConfig, VLMConfig
from tfv.data.loaders import get_mmstar_loader
from tfv.data.processors import get_tokenizer
from tfv.models.vlm import VisionLanguageModel

device = torch.device("cuda")
tok = get_tokenizer(VLMConfig().lm_tokenizer)
batch = next(iter(get_mmstar_loader(EvalConfig(), VLMConfig(), limit=16)))
answers = tok.batch_decode(batch["labels"], skip_special_tokens=True)

for path in ["lusxvr/nanoVLM-222M", f"{CHECKPOINT_DIR}/{RUN_NAME}/best_mmstar", f"{CHECKPOINT_DIR}/{RUN_NAME}/final"]:
    model = VisionLanguageModel.from_pretrained(path).to(device).eval()
    torch.manual_seed(0)
    with torch.no_grad():
        gen = model.generate(batch["input_ids"].to(device), batch["images"].to(device), batch["attention_mask"].to(device))
    print(f"\n{path}")
    for answer, out in zip(answers, tok.batch_decode(gen, skip_special_tokens=True)):
        print(f"  correct {answer!r:6} | model {out!r}")


lusxvr/nanoVLM-222M
  correct 'A'    | model ' B\nThe suit'
  correct 'B'    | model ' D\n'
  correct 'D'    | model ' D\nAnswer:'
  correct 'C'    | model ' A  '
  correct 'A'    | model ' C\n'
  correct 'D'    | model ' A\nD.'
  correct 'A'    | model ' D\nAnswer:'
  correct 'D'    | model ' D\nAnswer'
  correct 'B'    | model ' B\nAnswer:'
  correct 'D'    | model ' D\nD.'
  correct 'D'    | model ' D!\n'
  correct 'D'    | model ' D\nAnswer:'
  correct 'A'    | model ' B   10'
  correct 'B'    | model ' B\nGold.'
  correct 'A'    | model ' D\nD.'
  correct 'C'    | model ' D\nAnswer:'

/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/best_mmstar
  correct 'A'    | model 'C: D.'
  correct 'B'    | model 'A: B.'
  correct 'D'    | model 'D: Home decoration'
  correct 'C'    | model 'D: D.'
  correct 'A'    | model 'D: D.'
  correct 'D'    | model 'D: B.'
  correct 'A'    | model 'A: C.'
  correct 'D'    | model 'C: D.'
  correct 'B'    | model 'A: A.'
  correct 'D'   

The published model answers with a letter after a space (`" D"`). Ours writes a letter, a colon and often a second letter or a short phrase (`"C: D."`), and `final` leans heavily toward D. All three score 8 out of 16 here, too few questions to separate them.

### Padding Visible

The only approved change to nanoVLM's model code merges the causal and padding masks. If nanoVLM had trained with the padding visible, that would explain its higher validation loss, so this recomputes it with no padding mask.

In [19]:
from torch.utils.data import DataLoader
from tfv.train.train import count_targets, to_device

@torch.no_grad()
def val_loss_no_padding_mask(model: VisionLanguageModel, loader: DataLoader) -> float:
    """
    Validation loss with no attention mask, so the answer tokens can attend to the padding.

    Same token-weighted average as `evaluate_val_loss`, but `attention_mask` is passed as None.

    Args:
        model: The VLM to score.
        loader: Validation loader yielding `VQACollator` batches.
    Returns:
        float: Mean loss per answer token over the whole loader.
    """
    model.eval()
    loss_sum, n = 0.0, 0
    for batch in loader:
        batch = to_device(batch, device)
        k = count_targets(batch["labels"])
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = model(batch["input_ids"], batch["image"], None, targets=batch["labels"])  # Padding visible
        loss_sum += loss.float().item() * k
        n += k
    return loss_sum / n

for path in ["lusxvr/nanoVLM-222M", f"{CHECKPOINT_DIR}/{RUN_NAME}/best_mmstar", f"{CHECKPOINT_DIR}/{RUN_NAME}/final"]:
    model = VisionLanguageModel.from_pretrained(path).to(device)
    print(f"{path}: val loss, padding visible {val_loss_no_padding_mask(model, val_loader):.4f}")

lusxvr/nanoVLM-222M: val loss, padding visible 2.7017
/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/best_mmstar: val loss, padding visible 1.7190
/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/final: val loss, padding visible 1.6677


Every model gets worse, the published one most (1.90 to **2.70**), so nanoVLM trained with the padding masked, as we do.

### Space Before Answer

The published checkpoint was uploaded two weeks before the `v0.1` tag, and nanoVLM's first commit trained with a space before the answer (`Answer: Yes`), which the tag removed. This recomputes validation loss with a space before every answer.

In [20]:
from torch.utils.data import DataLoader, Dataset
from tfv.data.collators import VQACollator
from tfv.data.processors import get_tokenizer
from tfv.train.train import evaluate_val_loss

class SpaceBeforeAnswer(Dataset):
    """
    Wraps a VQA dataset and adds a space before every answer, so the training text reads
    `Answer: Yes` (nanoVLM's first commit) instead of `Answer:Yes` (v0.1).
    """

    def __init__(self, ds: Dataset) -> None:
        """
        Args:
            ds: The VQA dataset to wrap; its samples are dicts with an `answer` string.
        """
        self.ds = ds

    def __len__(self) -> int:
        """
        Returns:
            int: Number of samples in the wrapped dataset.
        """
        return len(self.ds)

    def __getitem__(self, i: int) -> dict:
        """
        Return sample i with a space prepended to its answer.

        Args:
            i: Index of the sample.
        Returns:
            dict: A copy of the sample, with `answer` starting with a space.
        """
        sample = dict(self.ds[i])
        sample["answer"] = " " + sample["answer"]
        return sample

tok = get_tokenizer(VLMConfig().lm_tokenizer)
spaced_loader = DataLoader(SpaceBeforeAnswer(val_ds), batch_size=256, num_workers=10,
                            collate_fn=VQACollator(tok, VLMConfig().lm_max_length))
for path in ["lusxvr/nanoVLM-222M", f"{CHECKPOINT_DIR}/{RUN_NAME}/best_mmstar", f"{CHECKPOINT_DIR}/{RUN_NAME}/final"]:
    model = VisionLanguageModel.from_pretrained(path).to(device)
    print(f"{path}: val loss, space before answer {evaluate_val_loss(model, spaced_loader, device, torch.bfloat16, True):.4f}")

lusxvr/nanoVLM-222M: val loss, space before answer 1.0102
/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/best_mmstar: val loss, space before answer 2.2459
/content/drive/MyDrive/tinyforge-vision/checkpoints/tfv-full-v1/final: val loss, space before answer 2.2380


The losses swap: the published checkpoint drops to **1.01** and ours rise to 2.24-2.25. Each model is good only in the format it was trained in, so the published checkpoint was trained with the space, and our training matches it in its own format (0.99 against 1.01). `WRITEUP.md` explains why the space most likely costs about 10 MMStar points.

## Upload

`best_mmstar`, the like-for-like comparison with the published checkpoint, pushed to the Hub as `Juanillaberia/TinyForge-Vision-222M`.

In [21]:
from tfv.models.vlm import VisionLanguageModel

VisionLanguageModel.from_pretrained(f"{CHECKPOINT_DIR}/{RUN_NAME}/best_mmstar").push_to_hub("Juanillaberia/TinyForge-Vision-222M", private=True)

Created repo:  https://huggingface.co/Juanillaberia/TinyForge-Vision-222M


CommitInfo(commit_url='https://huggingface.co/Juanillaberia/TinyForge-Vision-222M/commit/f8e537437f0a2b5ce7d95ef780151a59832faa33', commit_message='Upload TinyForge-Vision using push_to_hub', commit_description='', oid='f8e537437f0a2b5ce7d95ef780151a59832faa33', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Juanillaberia/TinyForge-Vision-222M', endpoint='https://huggingface.co', repo_type='model', repo_id='Juanillaberia/TinyForge-Vision-222M'), pr_revision=None, pr_num=None)